# Kubeflow Trainer

In [50]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("../.env-mlflow")

True

## Train function

In [44]:
def train_pytorch():
    import os
    import mlflow
    import mlflow.pytorch
    import torch
    from torch import nn
    import torch.nn.functional as F

    from torchvision import datasets, transforms
    import torch.distributed as dist
    from torch.utils.data import DataLoader, DistributedSampler

    # --- MLflow: Start run and get rank ---
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    global_rank = int(os.getenv("RANK", 0)) # Assuming RANK environment variable is set in DDP
    is_rank_0 = global_rank == 0

    if is_rank_0:
        # Start MLflow run only on rank 0
        mlflow.set_experiment("PyTorch-DDP-FashionMNIST")
        mlflow.start_run()
        print("MLflow Run Started on Rank 0.")
        
    # [1] Configure CPU/GPU device and distributed backend.
    # Kubeflow Trainer will automatically configure the distributed environment.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    # device, backend = ("cpu", "gloo")
    dist.init_process_group(backend=backend)

    print(
        "Distributed Training with WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}.".format(
            dist.get_world_size(),
            global_rank,
            local_rank,
        )
    )

    # [2] Define PyTorch CNN Model to be trained.
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # [3] Attach model to the correct device.
    device = torch.device(f"{device}:{local_rank}")
    model = nn.parallel.DistributedDataParallel(Net().to(device), device_ids=[local_rank] if device.type == 'cuda' else None) # Added device_ids for clarity/safety
    model.train()
    
    lr = 0.1 # Parameter to log
    momentum = 0.9 # Parameter to log
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    epochs = 3
    batch_size = 100
    
    # --- MLflow: Log parameters on rank 0 ---
    if is_rank_0:
        mlflow.log_param("learning_rate", lr)
        mlflow.log_param("momentum", momentum)
        mlflow.log_param("epochs", epochs)
        mlflow.log_param("batch_size", batch_size)
        mlflow.log_param("device", device.type)
        mlflow.log_param("world_size", dist.get_world_size())


    # [4] Get the Fashion-MNIST dataset and distributed it across all available devices.
    dataset = datasets.FashionMNIST(
        "./data",
        train=True,
        download=True,
        transform=transforms.Compose([transforms.ToTensor()]),
    )
    train_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=DistributedSampler(dataset),
    )

    # [5] Define the training loop.
    for epoch in range(epochs):
        # Set epoch for DistributedSampler
        train_loader.sampler.set_epoch(epoch)
        
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Attach tensors to the device.
            inputs, labels = inputs.to(device), labels.to(device)

            # Forward pass
            outputs = model(inputs)
            loss = F.nll_loss(outputs, labels)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # --- MLflow: Log batch loss on rank 0 ---
            if is_rank_0 and batch_idx % 10 == 0:
                step = epoch * len(train_loader) + batch_idx
                mlflow.log_metric("train_batch_loss", loss.item(), step=step)
                
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(train_loader.dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )
        
        # --- MLflow: Log epoch loss (could aggregate if needed, but for simplicity, log the last batch loss of the epoch) ---
        if is_rank_0:
            mlflow.log_metric("train_epoch_loss", loss.item(), step=epoch)


    # Wait for the training to complete and destroy to PyTorch distributed process group.
    dist.barrier()
    
    # --- MLflow: Log final model on rank 0 ---
    if is_rank_0:
        # Get the underlying model from DDP wrapper
        unwrapped_model = model.module
        
        # Log the PyTorch model
        mlflow.pytorch.log_model(
            pytorch_model=unwrapped_model,
            artifact_path="model",
            registered_model_name="FashionMNIST_CNN_DDP" # Optional: Register model
        )
        print("MLflow Model Logged on Rank 0.")
        
        # End the MLflow run
        mlflow.end_run()
        print("MLflow Run Ended on Rank 0.")

    if dist.get_rank() == 0:
        print("Training is finished")
        
    dist.destroy_process_group()

## Trainer configuration and process (runtimes)

In [45]:
from kubeflow.trainer import TrainerClient, CustomTrainer

# Initialize the client (assuming you are running this where it can connect to the cluster)
client = TrainerClient()

# List the available ClusterTrainingRuntime resources
print("Available ClusterTrainingRuntimes:")
for r in client.list_runtimes():
    # 'r.name' will be the name of the ClusterTrainingRuntime, e.g., 'torch-distributed'
    print(f"\t Runtime: {r.name}")

Available ClusterTrainingRuntimes:
	 Runtime: deepspeed-distributed
	 Runtime: mlx-distributed
	 Runtime: mpi-distributed
	 Runtime: torch-distributed
	 Runtime: torchtune-llama3.2-1b
	 Runtime: torchtune-llama3.2-3b


In [53]:
# 1. Launch the job - it will fail, but the parent resource is created
job_id = TrainerClient().train(
    trainer=CustomTrainer(
        func=train_pytorch,
        num_nodes=1,
        resources_per_node={
            "cpu": 5,
            "memory": "16Gi",
            "gpu": 1
        },
        packages_to_install=["mlflow", "boto3", "s3fs"],
        env={
            # "OMP_NUM_THREADS": "5",
            "AWS_ACCESS_KEY_ID":os.environ["AWS_ACCESS_KEY_ID"]
            "AWS_SECRET_ACCESS_KEY":os.environ["AWS_SECRET_ACCESS_KEY"]
            "MLFLOW_S3_ENDPOINT_URL":os.environ["MLFLOW_S3_ENDPOINT_URL"]
            "MLFLOW_TRACKING_URI":os.environ["MLFLOW_TRACKING_URI"]
        },
    ),
    runtime=TrainerClient().get_runtime("torch-distributed"),
)


In [77]:
for s in TrainerClient().get_job(name=job_id).steps:
    print(f"Step: {s.name}, Status: {s.status}, Devices: {s.device} x {s.device_count}")

Step: node-0, Status: Succeeded, Devices: gpu x 1


In [76]:
for logline in TrainerClient().get_job_logs(TrainerClient().get_job(name=job_id).name, follow=True):
    print(logline)

2025/09/28 23:07:46 INFO mlflow.tracking.fluent: Experiment with name 'PyTorch-DDP-FashionMNIST' does not exist. Creating a new experiment.
2025/09/28 23:07:46 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)
All git commands will error until this is rectified.
This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception
Example:
    

## List and delete jobs is it's needed

In [ ]:
TrainerClient().list_jobs()

In [33]:
client = TrainerClient() # Ensure this client is configured for your cluster/namespace

job_ids_to_delete = [
    job_id
]

for job_id in job_ids_to_delete:
    print(f"Deleting Kubeflow training job: {job_id}")
    try:
        # The job kind is usually inferred or specified implicitly by the runtime,
        # but delete_job is designed to find and delete the parent resource (TrainJob/PyTorchJob).
        client.delete_job(name=job_id)
        print(f"Successfully requested deletion for {job_id}")
    except Exception as e:
        print(f"Error deleting job {job_id}: {e}")

Deleting Kubeflow training job: he64e4e40db7
Error deleting job he64e4e40db7: Failed to delete TrainJob: kubeflow-user-example-com/he64e4e40db7
Deleting Kubeflow training job: yaac3511f88e
Successfully requested deletion for yaac3511f88e
